In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/mrdwin/groupe5-allstate-appautomatque-l2-esgcvak/a_estimer.csv
/kaggle/input/datasets/mrdwin/groupe5-allstate-appautomatque-l2-esgcvak/train.csv


In [2]:
#charger et compter
import pandas as pd

DOSSIER = "/kaggle/input/datasets/mrdwin/groupe5-allstate-appautomatque-l2-esgcvak/"

train = pd.read_csv(DOSSIER + "train.csv")
test  = pd.read_csv(DOSSIER + "a_estimer.csv")

print(train.shape, test.shape)                    # attendu : (150654, 132) (37664, 131)
print(set(train.columns) - set(test.columns))     # attendu : {'loss'}
print(train["id"].is_unique, test["id"].is_unique)
print(len(set(train["id"]) & set(test["id"])))    # attendu : 0 (aucun id commun)

(150654, 132) (37664, 131)
{'loss'}
True True
0


In [3]:
#Typer
cat_cols  = [c for c in train.columns if c.startswith("cat")]
cont_cols = [c for c in train.columns if c.startswith("cont")]
print(len(cat_cols), len(cont_cols))    # attendu : 116 et 14

for c in cat_cols:
    train[c] = train[c].astype("category")
print(train[cat_cols].nunique().agg(["min", "max"]))   # attendu : 2 et 326

116 14
min      2
max    317
dtype: int64


In [4]:
#Valeurs impossibles
import numpy as np

# Si l'étape 2 n'est pas faite : définir les listes de colonnes
cat_cols  = [c for c in train.columns if c.startswith("cat")]
cont_cols = [c for c in train.columns if c.startswith("cont")]

# 1. Valeurs manquantes (confirmer le "zéro manquant")
print("NaN train :", train.isna().sum().sum())
print("NaN test  :", test.isna().sum().sum())

# 2. La cible : montants positifs ?
print("loss <= 0 :", (train["loss"] <= 0).sum())
print("loss min / max :", train["loss"].min(), train["loss"].max())

# 3. Infinis dans les variables continues
print("inf train :", np.isinf(train[cont_cols]).sum().sum())
print("inf test  :", np.isinf(test[cont_cols]).sum().sum())

# 4. Modalités vides ou blanches dans les catégorielles
for nom, df in [("train", train), ("test", test)]:
    vides = sum((df[c].astype(str).str.strip() == "").sum() for c in cat_cols)
    print("modalités vides", nom, ":", vides)

# 5. Plage des continues : train vs test
plage = pd.DataFrame({
    "train_min": train[cont_cols].min(),
    "train_max": train[cont_cols].max(),
    "test_min":  test[cont_cols].min(),
    "test_max":  test[cont_cols].max(),
})
print(plage)

# 6. Continues du test hors de la plage du train
hors = {}
for c in cont_cols:
    n = ((test[c] < train[c].min()) | (test[c] > train[c].max())).sum()
    if n > 0:
        hors[c] = n
print("continues avec valeurs hors plage :", hors)

NaN train : 0
NaN test  : 0
loss <= 0 : 0
loss min / max : 0.67 121012.25
inf train : 0
inf test  : 0
modalités vides train : 0
modalités vides test : 0
        train_min  train_max  test_min  test_max
cont1    0.000016   0.984975  0.000016  0.984452
cont2    0.001149   0.862654  0.001149  0.862654
cont3    0.002634   0.944251  0.002634  0.944251
cont4    0.176921   0.954297  0.176921  0.952482
cont5    0.281143   0.982520  0.281143  0.983674
cont6    0.012683   0.997162  0.012683  0.997162
cont7    0.069503   1.000000  0.069503  1.000000
cont8    0.236880   0.980200  0.236880  0.980200
cont9    0.000080   0.995400  0.000080  0.993790
cont10   0.000000   0.994980  0.000000  0.994630
cont11   0.035321   0.998742  0.035321  0.996799
cont12   0.036232   0.998484  0.036232  0.997211
cont13   0.000228   0.988494  0.000228  0.982479
cont14   0.180268   0.844848  0.179722  0.844729
continues avec valeurs hors plage : {'cont5': np.int64(2), 'cont14': np.int64(1)}


In [5]:
#le plancher
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error as mae

tr, va = train_test_split(train, test_size=0.2, random_state=0)
for nom, c in [("moyenne", tr["loss"].mean()), ("médiane", tr["loss"].median())]:
    print(nom, round(c, 2), "MAE =", round(mae(va["loss"], [c]*len(va)), 2))

moyenne 3040.66 MAE = 1964.44
médiane 2119.07 MAE = 1806.01


Séance: S1
Nom :AGON KINHOU Fifamè Nathalie Ariane
Porteur:P2
Etapes Assignés:1.2 --> 1.3 --> 1.5 --> 1.10